In [5]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import tkinter as tk
from tkinter import filedialog

# Fix rendering
pio.renderers.default = "browser"

# ---------------------------
# 1. Pick CSV
# ---------------------------
def pick_csv():
    root = tk.Tk()
    root.withdraw()
    return filedialog.askopenfilename(
        title="Select CSV file",
        filetypes=[("CSV files", "*.csv")]
    )

file_path = pick_csv()

if not file_path:
    print("❌ No file selected")
    exit()

# ---------------------------
# 2. Load data
# ---------------------------
df = pd.read_csv(file_path)

# Clean duplicate Year column
if "Year" in df.columns:
    df = df.drop(columns=["Year"])

print("\n✅ Loaded:", file_path)

# ---------------------------
# 3. Show columns
# ---------------------------
cols = list(df.columns)

print("\nAvailable columns:\n")
for i, col in enumerate(cols, 1):
    print(f"{i}. {col}")

# ---------------------------
# 4. Pick X/Y
# ---------------------------
x_choice = int(input("\nSelect X-axis column number: "))
y_choice = int(input("Select Y-axis column number: "))

x_col = cols[x_choice - 1]
y_col = cols[y_choice - 1]

# ---------------------------
# 5. Colour grouping
# ---------------------------
use_colour = input("\nSplit by category? (y/n): ").strip().lower()

if use_colour == "y":
    c_choice = int(input("Select column number for colour grouping: "))
    color_col = cols[c_choice - 1]
else:
    color_col = None

# ---------------------------
# 6. SMART DATA FIX
# ---------------------------
group_cols = [x_col]

if color_col:
    group_cols.append(color_col)

# ✅ If duplicates exist → aggregate
if df.groupby(group_cols).size().max() > 1:
    print("\n⚠️ Aggregating data (multiple rows per group detected)")
    df_plot = df.groupby(group_cols, as_index=False)[y_col].sum()
else:
    df_plot = df.copy()

# ⚠️ If user picked "measure", suggest faceting instead
facet = None
if color_col == "measure":
    print("\n⚠️ Measures have different scales → using facet charts instead")
    facet = "measure"
    color_col = None  # remove colour to avoid confusion

# ---------------------------
# 7. PLOT
# ---------------------------
fig = px.line(
    df_plot,
    x=x_col,
    y=y_col,
    color=color_col,
    facet_col=facet,
    title=f"{y_col} vs {x_col}"
)

# ---------------------------
# 8. CLEAN FORMATTING
# ---------------------------
fig.update_layout(
    xaxis_title=x_col,
    yaxis_title=y_col,
    hovermode="x unified"
)

fig.show()


✅ Loaded: C:/Users/agallen002/OneDrive - PwC/Market Statistics/tidy_outputs/annex-private-motor-insurance-report-7__Figure23.csv

Available columns:

1. measure
2. year
3. value

⚠️ Measures have different scales → using facet charts instead
